---
title: Batch GitHub Repository Operations Using GitHub Rest API
created: 2026-04-05 11:04:25.437353
date: 2026-04-05 11:42:02.900456
authors:
- bendu
label: batch-github-repository-operations-using-github-rest-api
license: CC-BY-4.0
tags:
- computer science
- programming
- GitHub
- repository
- batch
- operation
- github-rest-api
---
**Things on this page are fragmentary and immature notes/thoughts of the author. Please read with your own judgement!**


In [2]:
import shutil
from pathlib import Path

from dulwich import porcelain
from dulwich.repo import Repo
from github_rest_api import Organization, Repository, RepositoryType

In [3]:
org = Organization(token="", owner="legendu-net")
org

## Get Non-archived Public Repositories

In [13]:
repos = org.get_repositories(RepositoryType.PUBLIC)
repos_active = [repo for repo in repos if not repo["archived"]]

In [14]:
len(repos_active)

65

## Clone Non-archived Repositories

In [18]:
for repo in repos_active:
    porcelain.clone(repo["ssh_url"])

## Update GitHub Actions Workflow

In [23]:
!ls -lha aiutil.git/.github/

total 0
drwxrwxr-x+ 1 dclong docker  18 Jan 20 12:20 .
drwxrwxr-x+ 1 dclong docker 154 Jan 20 12:20 ..
drwxrwxr-x 1 dclong docker 156 Jan 20 12:20 workflows


In [24]:
!ls -lha aiutil.git/.github/workflows/

total 24K
drwxrwxr-x+ 1 dclong docker 156 Jan 20 12:20 .
drwxrwxr-x+ 1 dclong docker  18 Jan 20 12:20 ..
-rw-r--r-- 1 dclong docker 495 Jan 20 12:20 automerge.yml
-rw-r--r-- 1 dclong docker 637 Jan 20 12:20 create_pull_request.yml
-rw-r--r-- 1 dclong docker 609 Jan 20 12:20 lint.yml
-rw-r--r-- 1 dclong docker 720 Jan 20 12:20 pre-release.yml
-rw-r--r-- 1 dclong docker 546 Jan 20 12:20 release.yml
-rw-r--r-- 1 dclong docker 907 Jan 20 12:20 test.yml


In [54]:
def update_workflow(path: Path) -> None:
    if path.name == "docker-jupyterlab.git":
        return
    dir_wf = path / ".github/workflows"
    if not dir_wf.is_dir():
        return
    file = dir_wf / "create_pull_request.yml"
    if file.is_file():
        file.unlink()
    shutil.copy2(
        "docker-jupyterlab.git/.github/workflows/create_pr_dev_to_main.yml", dir_wf
    )


def push_changes(path):
    repo = Repo(path)
    status = porcelain.status(repo)
    if status.untracked or status.unstaged:
        porcelain.add(path)
        porcelain.commit(path, message="update workflows")
        porcelain.push(Repo(path))

In [38]:
for path in Path().glob("*.git/"):
    update_workflow(path)

In [55]:
for path in Path().glob("*.git/"):
    push_changes(path)